# 🦈 Park Shark × 🪰 Fly Brain — the fly picks your parking lot

**Run the organizers' `fly_brain_starter.ipynb` cells first** (install → load brain → pick neurons → simulator). These cells reuse its `simulate`, `rate`, `sugar`, `bitter`, `looming`, `mn9`, `giant_fibre`, `mdn`, `W`, `N`.

The story: an NC State student is a hungry fly. Each candidate lot is a food source. Our Park Shark estimator (no sensors yet — model estimates) turns each lot into tastes:
- **sugar** = chance of an open spot you can legally use, sweeter when the walk is short
- **bitter** = permit risk + price
- **looming** = the lot is filling fast / nearly full — a threat rushing at you

The real 139k-neuron connectome then decides per lot: **eat** (MN9 = commit), **back off** (MDN), or **flee** (Giant Fibre = divert now).

In [ ]:
# Upload senses.json (made by build_ncstate.py)
import json
from google.colab import files
up = files.upload()
senses = json.loads(next(iter(up.values())))
print(senses['scenario'], '|', senses['persona'])
print(len(senses['lots']), 'candidate lots')

In [ ]:
# Three fixed-seed brain runs per lot, reset from rest each time.
import pandas as pd
import numpy as np
# Calibrated on the measured v783 response surface; engineering thresholds.
GF_DIVERT_HZ, MN9_MIN_HZ = 130, 5
def decide(lot, W=W):
    inputs = [(sugar, lot['sugar_hz']), (bitter, lot['bitter_hz']), (looming, lot['looming_hz'])]
    inputs = [(g, hz) for g, hz in inputs if hz > 0]
    counts = [simulate(inputs, ms=200, W=W, seed=seed) for seed in [0, 1, 2]]
    eat, flee, back = [float(np.mean([rate(c, group) for c in counts])) for group in [mn9, giant_fibre, mdn]]
    verdict = 'FLEE 🏃' if flee >= GF_DIVERT_HZ else ('BACK OFF ↩️' if back > eat or eat < MN9_MIN_HZ else 'EAT 🌭')
    return {'lot': lot['lot'], 'walk_min': lot['walk_min'],
            'sugar': lot['sugar_hz'], 'bitter': lot['bitter_hz'], 'loom': lot['looming_hz'],
            'eat_mn9_hz': round(eat, 1), 'flee_gf_hz': round(flee, 1), 'back_mdn_hz': round(back, 1),
            'verdict': verdict}

rows = [decide(l) for l in senses['lots']]
df = pd.DataFrame(rows).sort_values('eat_mn9_hz', ascending=False).reset_index(drop=True)
pick = df[df.verdict.str.startswith('EAT')].head(1)
print('\n🪰 the fly picks:', pick.lot.iloc[0] if len(pick) else 'nothing edible — it flees everything')
df

In [ ]:
# Dumb baseline (the handbook demands one): pick by plain score, no neurons.
base = max(senses['lots'], key=lambda l: l['sugar_hz'] - l['bitter_hz'] - l['looming_hz'])
fly_pick = pick.lot.iloc[0] if len(pick) else None
print('baseline picks :', base['lot'])
print('fly brain picks:', fly_pick)
print('agree' if base['lot'] == fly_pick else 'DISAGREE — discuss why in the demo: the fly is a fixed nonlinear filter, not an optimizer')

In [ ]:
# Lesion lab: cut the Giant Fibre's INPUTS — the fly loses its 'bail out' reflex
# Its rate becomes zero. Other circuits can still cause BACK OFF: a behavioral
# switch is measured, never guaranteed for every input.
import numpy as np, scipy.sparse as sp
keep = np.ones(N); keep[giant_fibre] = 0
W_lesion = (W @ sp.diags(keep)).tocsr()
worst = max(senses['lots'], key=lambda l: l['looming_hz'])
print('most threatening lot:', worst['lot'])
print('intact  :', decide(worst)['verdict'])
print('lesioned:', decide(worst, W=W_lesion)['verdict'])

In [ ]:
# Export for the Park Shark map (drop next to web/data.json)
out = {'scenario': senses['scenario'], 'persona': senses['persona'], 'decisions': rows}
json.dump(out, open('decisions.json', 'w'), indent=1)
files.download('decisions.json')

**Limitations to say out loud in the demo:** occupancy is a model estimate (assumed hour templates, wide bands — no sensor has ever reported); permit rules are simplified; the fly is a fixed filter, so we compare it against a dumb baseline instead of claiming it is optimal. **Next step:** Park Shark floor sensors replace the estimate with measured spots and the same senses sharpen.